In [ ]:
import gradio as gr
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import ctransformers
from langchain.prompts import PromptTemplate
from langchain.chains.retrieval_qa.base import RetrievalQA
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from transformers import AutoTokenizer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pdf2image import convert_from_path
import easyocr
import torch
import os
import numpy as np
from PIL import Image
import shutil
import time
import threading

DATA_PATH = 'D:/ocr/data/'
TEMP_FOLDER = os.path.join(DATA_PATH, "TEMP")
Img_folder=os.path.join(TEMP_FOLDER, "images")

PDF_DB_PATH = 'vectorstore/db_pdf'
IMAGE_DB_PATH = 'vectorstore/db_image'
MERGED_DB_PATH = 'vectorstore/merged_db'

# check directories exist
os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(os.path.dirname(PDF_DB_PATH), exist_ok=True)
os.makedirs(os.path.dirname(IMAGE_DB_PATH), exist_ok=True)
os.makedirs(os.path.dirname(MERGED_DB_PATH), exist_ok=True)
os.makedirs(Img_folder,exist_ok=True)

# Device setup for LLM
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# EasyOCR initialization
reader = easyocr.Reader(['en'])

# Custom prompt template for chatbot
custom_prompt_template = """You are a sophisticated chatbot designed to provide detailed and accurate information based on the mamaogram report provided.
Your goal is to assist users by answering their questions, offering insights, and providing recommendations based on the merged content of these sources.
If you don't know the answer, just say that you don't know, and don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

# Function to create custom prompt for QA
def set_custom_prompt():
    return PromptTemplate(template=custom_prompt_template, input_variables=['context', 'question'])

# Load LLM
def load_llm():
    llm = ctransformers.CTransformers(
        model="D:/ocr/mistral-7b-instruct-v0.2.Q5_K_M.gguf",
        model_type="mistral",
        max_new_tokens=512,
        temperature=0.5
    )
    return llm


def create_vector_db(pdf_path):
    try:
        os.makedirs(TEMP_FOLDER, exist_ok=True)

        pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
        output_folder = os.path.join(TEMP_FOLDER, pdf_name)
        os.makedirs(output_folder, exist_ok=True)

        loader = PyPDFLoader(pdf_path)
        documents = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
        texts = text_splitter.split_documents(documents)

        embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', model_kwargs={'device': 'cpu'})
        db = FAISS.from_documents(texts, embeddings)
        db.save_local(PDF_DB_PATH)

        # Convert PDF pages to images
        images = convert_from_path(pdf_path)
        image_paths = []
        for i, img in enumerate(images):
            img_path = os.path.join(output_folder, f'page_{i + 1}.png')
            img.save(img_path, 'PNG')
            image_paths.append(img_path)
            
        merge_message = merge_vector_stores()

        return f"Successfully vectorized {len(images)} pages. {merge_message}", image_paths
    except Exception as e:
        return f"Error processing PDF: {e}", []
        

# Function to extract text from an image
def extract_text_from_image(image):
    image_array = np.array(image)
    result = reader.readtext(image_array)
    return " ".join([text for _, text, _ in result])

def vectorize_image_text(image_text):
    splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20) 
    text_chunks = splitter.split_text(image_text) 
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', model_kwargs={'device': 'cpu'})
    
    # Create a vector database from the text chunks
    db = FAISS.from_texts(text_chunks, embeddings)
    db.save_local(IMAGE_DB_PATH)  # Save Image vector store
    return merge_vector_stores()

# Function to merge PDF and image vector stores
def merge_vector_stores():
    try:
        pdf_exists = os.path.exists(PDF_DB_PATH)
        image_exists = os.path.exists(IMAGE_DB_PATH)

        if not pdf_exists and not image_exists:
            return "No vector store exists to merge."

        embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', model_kwargs={'device': 'cpu'})

        # Load existing vector stores
        db_pdf = FAISS.load_local(PDF_DB_PATH, embeddings, allow_dangerous_deserialization=True) if pdf_exists else None
        db_image = FAISS.load_local(IMAGE_DB_PATH, embeddings, allow_dangerous_deserialization=True) if image_exists else None

        # Merge logic
        if db_pdf and db_image:
            db_pdf.merge_from(db_image)
            db_pdf.save_local(MERGED_DB_PATH)
            return f"Merged vector stores successfully into {MERGED_DB_PATH}."
        elif db_pdf:
            db_pdf.save_local(MERGED_DB_PATH)
            return f"Only PDF vector store exists. Saved PDF store into {MERGED_DB_PATH}."
        elif db_image:
            db_image.save_local(MERGED_DB_PATH)
            return f"Only Image vector store exists. Saved Image store into {MERGED_DB_PATH}."
        else:
            return "No vector store to merge."
    except Exception as e:
        return f"Error merging vector stores: {e}"
    

# Load the tokenizer for your specific model
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def chunk_text(text, max_tokens=512):
    """Split the text into chunks that are no longer than max_tokens."""
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk = tokens[i:i + max_tokens]
        chunks.append(tokenizer.decode(chunk, skip_special_tokens=True))
    return chunks

def final_result(query):
    try:
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={'device': device})
        db = FAISS.load_local(MERGED_DB_PATH, embeddings, allow_dangerous_deserialization=True)
        llm = load_llm()

        chunks = chunk_text(query)
        results = []
        for chunk in chunks:
            qa = RetrievalQA.from_chain_type(
                llm=llm,
                chain_type='stuff',
                retriever=db.as_retriever(search_kwargs={'k': 1}),
                return_source_documents=True
            )
            response = qa({'query': chunk})
            results.append(response['result'])

        # combine results from all chunks
        final_response = " ".join(results)
        return final_response  
    except Exception as e:
        return f"Error with LLM or vector store: {e}"


def clear_vectorstores():
    try:
        # Clear vector stores
        if os.path.exists(PDF_DB_PATH):
            shutil.rmtree(PDF_DB_PATH)
        
        if os.path.exists(IMAGE_DB_PATH):
            shutil.rmtree(IMAGE_DB_PATH)

        if os.path.exists(MERGED_DB_PATH):
            shutil.rmtree(MERGED_DB_PATH)

        data_folder = 'D:\\ocr\\data'
        if os.path.exists(data_folder):
            shutil.rmtree(data_folder)  
            os.makedirs(data_folder)  

        return "Vector stores and data folder cleared successfully."
    except Exception as e:
        return f"Error clearing vector stores or data folder: {e}"

def move_image_to_data_folder(image_path, from_explorer=False):
    if from_explorer:
        return "Image remains in its original location as it was selected from the file explorer."
    
    try:
        target_path = os.path.join(Img_folder, os.path.basename(image_path))
        shutil.move(image_path, target_path)
        return f"Moved image to {target_path}."
    except Exception as e:
        return f"Error moving image: {e}"

def handle_file_upload(file_paths, from_explorer=False):
    status_messages = []
    all_images = []

    if file_paths is None: 
        return None, "No files selected.", [], gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)

    for file_path in file_paths:
        if file_path.endswith('.pdf'):
            status, images = create_vector_db(file_path)
            status_messages.append(status)
            all_images.extend(images)
        else:
            try:
                image = Image.open(file_path)
                all_images.append(image)

                # Vectorization happens after displaying the image
                extracted_text = extract_text_from_image(image)
                merge_message = vectorize_image_text(extracted_text)
                status_messages.append(f"Image processed: {merge_message}")

                move_message = move_image_to_data_folder(file_path, from_explorer)
                status_messages.append(move_message)

            except Exception as e:
                status_messages.append(f"Error processing image {file_path}: {e}")

    merge_message = merge_vector_stores()
    return None, "\n".join(status_messages), all_images, gr.update(visible=False), gr.update(visible=True), gr.update(visible=True)

def list_files(directory):
    try:
        return [f for f in os.listdir(directory) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    except Exception as e:
        print(f"Error listing files: {e}")
        return []

def handle_directory_change():
    files = list_files(DATA_PATH)
    file_paths = [os.path.join(DATA_PATH, f) for f in files]  
    file_explorer_update = gr.update(value=file_paths) 
    status_text_update = gr.update(value=f"Found {len(files)} files.")  
    return file_explorer_update, status_text_update  

def periodic_refresh():
    while True:
        handle_directory_change()  
        time.sleep(5)  

with gr.Blocks() as app:
    with gr.Row():
        refresh_button = gr.Button("Refresh File Explorer")
        status_text = gr.Textbox(label="Status")
    file_explorer = gr.FileExplorer(label="Select Images", interactive=True) 


    refresh_button.click(
        handle_directory_change,
        inputs=[],
        outputs=[file_explorer, status_text]  
    )

threading.Thread(target=periodic_refresh, daemon=True).start()

def setup_interface():
    with gr.Blocks() as app:
        with gr.Tab("Upload PDFs or Images"):
            with gr.Row():
                with gr.Column(scale=1):
                    file_explorer_input = gr.FileExplorer(
                        label="Select PDFs or Images from Data Folder", 
                        root_dir=DATA_PATH,
                        file_count="multiple", 
                        min_width=200,
                        #glob="**/*.jpeg"
                        render=True
                    )
                    file_input = gr.File(
                        label="Or Upload PDFs or Images from Anywhere", 
                        file_count="multiple", 
                        min_width=200
                    )
                    clear_button = gr.Button("Clear Data & Reset UI", min_width=200)

                with gr.Column(scale=1):
                    image_preview = gr.Image(
                        label="Selected Image Preview", 
                        visible=True, 
                        min_width=50,
                        interactive=True
                    )
                    gallery_output = gr.Gallery(
                        label="Extracted Pages", 
                        show_label=True,
                        visible=False, 
                        min_width=250
                    )
                    vectorization_status = gr.Textbox(
                        label="Status", 
                        interactive=False, 
                        min_width=250
                    )

                with gr.Column(scale=3):
                    chatbox_output = gr.Chatbot(
                        label="ChatBot", 
                        placeholder="Your conversation will appear here...", 
                        min_width=500
                    )
                    question_input = gr.Textbox(
                        label="Ask a Question", 
                        placeholder="Type your question here...", 
                        visible=False, 
                        min_width=500
                    )
                    submit_button = gr.Button("Submit", visible=False, min_width=300)

            def handle_file_input(selected_files):
                if selected_files is None:
                    return None, "No files selected.", [], gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)

                file_paths = [file.name for file in selected_files]
                return handle_file_upload(file_paths)

            # Handle file uploads from the file explorer
            file_explorer_input.change(
                lambda file_paths: handle_file_upload(file_paths, from_explorer=True),
                inputs=file_explorer_input,
                outputs=[image_preview, vectorization_status, gallery_output, image_preview, question_input, submit_button]
            )

            # Handle file uploads from the file input
            file_input.change(
                handle_file_input, 
                inputs=file_input, 
                outputs=[image_preview, vectorization_status, gallery_output, image_preview, question_input, submit_button]
            )

            def handle_query_submission(query):
                answer = final_result(query)
                chat_history_state.append((query, answer))
                return chat_history_state  

            chat_history_state = []  
            submit_button.click(handle_query_submission, inputs=question_input, outputs=chatbox_output)

            def clear_all():
                clear_status = clear_vectorstores()
                chat_history_state.clear()  
                return (
                    [],  
                    "",  
                    gr.update(value=None,visible=True), 
                    gr.update(value=""), 
                    gr.update(visible=False), 
                    gr.update(value=[]),  
                    clear_status, 
                    gr.update(value=None), 
                    gr.update(value=None)   
                )

            clear_button.click(
                clear_all, 
                outputs=[
                    chatbox_output,          
                    vectorization_status,    
                    image_preview,           
                    question_input,          
                    submit_button,           
                    gallery_output,          
                    vectorization_status,    
                    file_explorer_input,     
                    file_input               
                ]
            )

        with gr.Tab("FAQs"):
            gr.Markdown(""" # Frequently Asked Questions

                **Q1: How do I remove an uploaded image if I selected the wrong one?**
                A: Just click on the "X" in the file upload options to remove the image you don't want. You can upload additional images or files after that.

                **Q3: What happens when I clear the data?**
                A: Clicking the "Reset App" button will reset the current session, clear the vector stores, and allow you to start a fresh process.

                **Q4: How do I ask the chatbot questions?**
                A: Once you upload documents and images, you'll be able to ask questions based on the content. Simply type your question into the input field and hit "Submit".
            """)

    app.launch()

if __name__ == "__main__":
    setup_interface()

d:\ocr\env\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


d:\ocr\env\Lib\site-packages\gradio\analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(
d:\ocr\env\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
d:\ocr\env\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
d:\ocr\env\Lib\site-packages\transformers\tok

In [ ]:
import gradio as gr
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import ctransformers
from langchain.prompts import PromptTemplate
from langchain.chains.retrieval_qa.base import RetrievalQA
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from transformers import AutoTokenizer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pdf2image import convert_from_path
import easyocr
import torch
import os
import numpy as np
from PIL import Image
import shutil


DATA_PATH = 'D:/ocr/data/'
TEMP_FOLDER = os.path.join(DATA_PATH, "TEMP")
Img_folder=os.path.join(TEMP_FOLDER, "images")

PDF_DB_PATH = 'vectorstore/db_pdf'
IMAGE_DB_PATH = 'vectorstore/db_image'
MERGED_DB_PATH = 'vectorstore/merged_db'

os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(os.path.dirname(PDF_DB_PATH), exist_ok=True)
os.makedirs(os.path.dirname(IMAGE_DB_PATH), exist_ok=True)
os.makedirs(os.path.dirname(MERGED_DB_PATH), exist_ok=True)



device = 'cuda' if torch.cuda.is_available() else 'cpu'
reader = easyocr.Reader(['en'])

# Custom prompt template for chatbot
custom_prompt_template = """You are a sophisticated chatbot designed to provide detailed and accurate information based on the data provided.
Your goal is to assist users by answering their questions, offering insights, and providing recommendations based on the merged content of these sources.
If you don't know the answer, just say that you don't know, and don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

# Function to create custom prompt for QA
def set_custom_prompt():
    return PromptTemplate(template=custom_prompt_template, input_variables=['context', 'question'])

# Load LLM
def load_llm():
    llm = ctransformers.CTransformers(
        model="D:/ocr/mistral-7b-instruct-v0.2.Q5_K_M.gguf",
        model_type="mistral",
        max_new_tokens=512,
        temperature=0.5
    )
    return llm


def create_vector_db(pdf_path):
    try:
        os.makedirs(TEMP_FOLDER, exist_ok=True)

        pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
        output_folder = os.path.join(TEMP_FOLDER, pdf_name)
        os.makedirs(output_folder, exist_ok=True)

        loader = PyPDFLoader(pdf_path)
        documents = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
        texts = text_splitter.split_documents(documents)

        embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', model_kwargs={'device': 'cpu'})
        db = FAISS.from_documents(texts, embeddings)
        db.save_local(PDF_DB_PATH)

        # Convert PDF pages to images
        images = convert_from_path(pdf_path)
        image_paths = []
        for i, img in enumerate(images):
            img_path = os.path.join(output_folder, f'page_{i + 1}.png')
            img.save(img_path, 'PNG')
            image_paths.append(img_path)
            
        merge_message = merge_vector_stores()

        return f"Successfully vectorized {len(images)} pages. {merge_message}", image_paths
    except Exception as e:
        return f"Error processing PDF: {e}", []
        

# Function to extract text from an image
def extract_text_from_image(image):
    image_array = np.array(image)
    result = reader.readtext(image_array)
    return " ".join([text for _, text, _ in result])

def vectorize_image_text(image_text):
    splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20) 
    text_chunks = splitter.split_text(image_text) 
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', model_kwargs={'device': 'cpu'})
    
    # Create a vector database from the text chunks
    db = FAISS.from_texts(text_chunks, embeddings)
    db.save_local(IMAGE_DB_PATH)  # Save Image vector store
    return merge_vector_stores()

# Function to merge PDF and image vector stores
def merge_vector_stores():
    try:
        pdf_exists = os.path.exists(PDF_DB_PATH)
        image_exists = os.path.exists(IMAGE_DB_PATH)

        if not pdf_exists and not image_exists:
            return "No vector store exists to merge."

        embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', model_kwargs={'device': 'cpu'})

        # Load existing vector stores
        db_pdf = FAISS.load_local(PDF_DB_PATH, embeddings, allow_dangerous_deserialization=True) if pdf_exists else None
        db_image = FAISS.load_local(IMAGE_DB_PATH, embeddings, allow_dangerous_deserialization=True) if image_exists else None

        # Merge logic
        if db_pdf and db_image:
            db_pdf.merge_from(db_image)
            db_pdf.save_local(MERGED_DB_PATH)
            return f"Merged vector stores successfully into {MERGED_DB_PATH}."
        elif db_pdf:
            db_pdf.save_local(MERGED_DB_PATH)
            return f"Only PDF vector store exists. Saved PDF store into {MERGED_DB_PATH}."
        elif db_image:
            db_image.save_local(MERGED_DB_PATH)
            return f"Only Image vector store exists. Saved Image store into {MERGED_DB_PATH}."
        else:
            return "No vector store to merge."
    except Exception as e:
        return f"Error merging vector stores: {e}"
    

# Load the tokenizer for your specific model
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def chunk_text(text, max_tokens=512):
    """Split the text into chunks that are no longer than max_tokens."""
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk = tokens[i:i + max_tokens]
        chunks.append(tokenizer.decode(chunk, skip_special_tokens=True))
    return chunks

def final_result(query):
    try:
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={'device': device})
        db = FAISS.load_local(MERGED_DB_PATH, embeddings, allow_dangerous_deserialization=True)
        llm = load_llm()

        chunks = chunk_text(query)
        results = []
        for chunk in chunks:
            qa = RetrievalQA.from_chain_type(
                llm=llm,
                chain_type='stuff',
                retriever=db.as_retriever(search_kwargs={'k': 1}),
                return_source_documents=True
            )
            response = qa({'query': chunk})
            results.append(response['result'])

        # combine results from all chunks
        final_response = " ".join(results)
        return final_response  
    except Exception as e:
        return f"Error with LLM or vector store: {e}"


def clear_vectorstores():
    try:
        # Clear vector stores
        if os.path.exists(PDF_DB_PATH):
            shutil.rmtree(PDF_DB_PATH)
        
        if os.path.exists(IMAGE_DB_PATH):
            shutil.rmtree(IMAGE_DB_PATH)

        if os.path.exists(MERGED_DB_PATH):
            shutil.rmtree(MERGED_DB_PATH)

        data_folder = 'D:\\ocr\\data'
        if os.path.exists(data_folder):
            shutil.rmtree(data_folder)  
            os.makedirs(data_folder)  

        return "Vector stores and data folder cleared successfully."
    except Exception as e:
        return f"Error clearing vector stores or data folder: {e}"


def move_image_to_data_folder(image_path, from_explorer=False):
    if from_explorer:
        return "Image remains in its original location as it was selected from the file explorer."

    try:
        os.makedirs(Img_folder, exist_ok=True)
        target_path = os.path.join(Img_folder, os.path.basename(image_path))

        shutil.move(image_path, target_path)
        return f"Moved image to {target_path}."
    except Exception as e:
        return f"Error moving image: {e}"

def handle_file_upload(file_paths, from_explorer=False):
    status_messages = []
    all_images = []

    if file_paths is None: 
        return None, "No files selected.", [], gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)

    for file_path in file_paths:
        if file_path.endswith('.pdf'):
            status, images = create_vector_db(file_path)
            status_messages.append(status)
            all_images.extend(images)
        else:
            try:
                image = Image.open(file_path)
                all_images.append(image)

                # Vectorization happens after displaying the image
                extracted_text = extract_text_from_image(image)
                merge_message = vectorize_image_text(extracted_text)
                status_messages.append(f"Image processed: {merge_message}")

                move_message = move_image_to_data_folder(file_path, from_explorer)
                status_messages.append(move_message)

            except Exception as e:
                status_messages.append(f"Error processing image {file_path}: {e}")

    merge_message = merge_vector_stores()
    return None, "\n".join(status_messages), all_images, gr.update(visible=False), gr.update(visible=True), gr.update(visible=True)


def setup_interface():
    with gr.Blocks() as app:
        with gr.Tab("Upload PDFs or Images"):
            with gr.Row():
                with gr.Column(scale=1):
                    file_explorer_input = gr.FileExplorer(
                        label="Select PDFs or Images from Data Folder", 
                        root_dir=DATA_PATH,
                        file_count="multiple", 
                        min_width=200,
                        glob="**/*.png",

                    )
                    file_input = gr.File(
                        label="Or Upload PDFs or Images from Anywhere", 
                        file_count="multiple", 
                        min_width=200
                    )
                    clear_button = gr.Button("Clear Data & Reset UI", min_width=200)

                with gr.Column(scale=1):
                    image_preview = gr.Image(
                        label="Selected Image Preview", 
                        visible=True, 
                        min_width=50,
                        interactive=True
                    )
                    gallery_output = gr.Gallery(
                        label="Extracted Pages", 
                        show_label=True,
                        visible=False, 
                        min_width=250
                    )
                    vectorization_status = gr.Textbox(
                        label="Status", 
                        interactive=False, 
                        min_width=250
                    )

                with gr.Column(scale=3):
                    chatbox_output = gr.Chatbot(
                        label="ChatBot", 
                        placeholder="Your conversation will appear here...", 
                        min_width=500
                    )
                    question_input = gr.Textbox(
                        label="Ask a Question", 
                        placeholder="Type your question here...", 
                        visible=False, 
                        min_width=500
                    )
                    submit_button = gr.Button("Submit", visible=False, min_width=300)

            def handle_file_input(selected_files):
                if selected_files is None:
                    return None, "No files selected.", [], gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)

                file_paths = [file.name for file in selected_files]
                return handle_file_upload(file_paths)

            # Handle file uploads from the file explorer
            file_explorer_input.change(
                lambda file_paths: handle_file_upload(file_paths, from_explorer=True),
                inputs=file_explorer_input,
                outputs=[image_preview, vectorization_status, gallery_output, image_preview, question_input, submit_button]
            )

            # Handle file uploads from the file input
            file_input.change(
                handle_file_input, 
                inputs=file_input, 
                outputs=[image_preview, vectorization_status, gallery_output, image_preview, question_input, submit_button]
            )

            def handle_query_submission(query):
                answer = final_result(query)
                chat_history_state.append((query, answer))
                return chat_history_state  

            chat_history_state = []  
            submit_button.click(handle_query_submission, inputs=question_input, outputs=chatbox_output)

            def clear_all():
                clear_status = clear_vectorstores()
                chat_history_state.clear()  
                return (
                    [],  
                    "",  
                    gr.update(value=None,visible=True), 
                    gr.update(value=""), 
                    gr.update(visible=False), 
                    gr.update(value=[]),  
                    clear_status, 
                    gr.update(value=None), 
                    gr.update(value=None)   
                )

            clear_button.click(
                clear_all, 
                outputs=[
                    chatbox_output,          
                    vectorization_status,    
                    image_preview,           
                    question_input,          
                    submit_button,           
                    gallery_output,          
                    vectorization_status,    
                    file_explorer_input,     
                    file_input               
                ]
            )

        with gr.Tab("FAQs"):
            gr.Markdown(""" # Frequently Asked Questions

                **Q1: How do I remove an uploaded image if I selected the wrong one?**
                A: Just click on the "X" in the file upload options to remove the image you don't want. You can upload additional images or files after that.

                **Q3: What happens when I clear the data?**
                A: Clicking the "Reset App" button will reset the current session, clear the vector stores, and allow you to start a fresh process.

                **Q4: How do I ask the chatbot questions?**
                A: Once you upload documents and images, you'll be able to ask questions based on the content. Simply type your question into the input field and hit "Submit".
            """)

    app.launch()

if __name__ == "__main__":
    setup_interface()

d:\ocr\env\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Running on local URL:  http://127.0.0.1:7875

To create a public link, set `share=True` in `launch()`.


d:\ocr\env\Lib\site-packages\gradio\analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(
ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "d:\ocr\env\Lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 406, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\ocr\env\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\ocr\env\Lib\site-packages\fastapi\applications.py", line 1054, in __call__
    await super().__call__(scope, receive, send)
  File "d:\ocr\env\Lib\site-packages\starlette\applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "d:\ocr\env\Lib\site-packages\starlette